# Voltametria cíclica por diferenças finitas, e o disco rotatório
**Tópicos Avançados em Ciências – Química Inorgânica Avançada · 2026.2 · Aula 02 (eletroquímica)**

Nesta atividade vamos:
1. **simular um voltamograma cíclico** resolvendo as leis de Fick com a condição de Nernst na superfície, e ver o 0,446 de Randles–Ševčík sair da conta;
2. ajustar a **reta de $i_p$ contra $\sqrt{v}$** aos picos medidos por Elgrishi et al. (2018) e comparar o $D$ obtido com o da literatura;
3. sobrepor a **curva simulada à medida** a 10 V/s;
4. analisar um conjunto de dados de **disco rotatório** (Levich e Koutecký–Levich).

Os dados de voltametria são do Material Suplementar de N. Elgrishi et al., *J. Chem. Educ.* **95**, 197 (2018), doi:10.1021/acs.jchemed.7b00361: 1 mM de ferroceno em acetonitrila com 0,25 M de [NBu₄][PF₆], carbono vítreo de 3 mm, 303 K. Os pontos foram digitalizados das Figs. S3.2 e S4.1.

Complete as células marcadas com `# COMPLETE`. Atividade formativa: não vale nota.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

F = 96485.33212      # C/mol, a constante de Faraday
R = 8.314462618      # J/(mol K)
T = 303.15           # K, a temperatura do experimento do artigo
A = 0.0707           # cm2, disco de carbono vítreo de 3 mm
c_bulk = 1.0e-6      # mol/cm3 (1 mM)
D_lit = 2.24e-5      # cm2/s, ferroceno em CH3CN, Wang e Compton (2010), citado no SI
RTF = R * T / F      # V
print(f"RT/F = {RTF*1e3:.2f} mV a {T} K")

## 1. O voltamograma por diferenças finitas

Em variáveis adimensionais (Bard e Faulkner, apêndice B):
- potencial: $\theta = (F/RT)(E - E^{\circ\prime})$;
- tempo: $\tau = (Fv/RT)\,t$, de modo que cada unidade de $\tau$ varre $RT/F$ de potencial;
- distância: $X = x/\sqrt{D\,RT/(Fv)}$;
- corrente: $\chi = i / \left[F A c^* \sqrt{FvD/RT}\right]$.

Nessas variáveis a segunda lei de Fick fica $\partial c/\partial\tau = \partial^2 c/\partial X^2$, e a condição de Nernst na superfície é $c_{\mathrm{Ox}}(0)/c_{\mathrm{Red}}(0) = e^{\theta}$. A corrente é o gradiente na superfície.

O esquema explícito: a cada passo, (i) impõe-se Nernst em $X = 0$ com a conservação do fluxo ($c_{\mathrm{Ox}}(0) + c_{\mathrm{Red}}(0) = c_{\mathrm{Ox}}(1) + c_{\mathrm{Red}}(1)$, para $D$ iguais), (ii) lê-se a corrente, (iii) difunde-se o interior com $c_j \leftarrow c_j + \lambda\,(c_{j+1} - 2c_j + c_{j-1})$, com $\lambda = \Delta\tau/\Delta X^2 < 1/2$.

Escrevemos a simulação para a **redução** de uma espécie Ox que parte de $c^*$ (é a convenção do livro); para a oxidação do ferroceno basta espelhar $E$ e $i$.

In [ ]:
def simular_cv(n_por_RTF=200, theta_max=24.0, lamb=0.45):
    dtau = 1.0 / n_por_RTF
    dX = np.sqrt(dtau / lamb)
    n_passos = int(round(4 * theta_max / dtau))        # ida e volta
    nX = int(6.0 * np.sqrt(n_passos * dtau) / dX) + 2
    cO = np.ones(nX)          # a espécie que é reduzida parte de c* em toda parte
    cR = np.zeros(nX)
    thetas = np.empty(n_passos)
    chis = np.empty(n_passos)
    theta = theta_max
    for k in range(n_passos):
        theta -= dtau if k < n_passos // 2 else -dtau   # desce, depois sobe
        # (i) Nernst na superfície: cO[0]/cR[0] = exp(theta), com cO[0] + cR[0] = cO[1] + cR[1]
        r = ...                                   # COMPLETE
        cR[0] = ...                               # COMPLETE
        cO[0] = ...                               # COMPLETE
        # (ii) a corrente é o gradiente de cO na superfície
        chis[k] = (-3.0 * cO[0] + 4.0 * cO[1] - cO[2]) / (2.0 * dX)
        thetas[k] = theta
        # (iii) difusão no interior
        cO[1:-1] += lamb * (cO[2:] - 2 * cO[1:-1] + cO[:-2])
        cR[1:-1] += lamb * (cR[2:] - 2 * cR[1:-1] + cR[:-2])
    return thetas, chis

theta, chi = simular_cv()
n = theta.size // 2
chi_p = chi[:n].max()
theta_p = theta[:n][chi[:n].argmax()]
theta_pv = theta[n:][chi[n:].argmin()]
print(f"pico adimensional chi_p = {chi_p:.4f}   (Nicholson e Shain, 1964: 0,4463)")
print(f"E_p - E0' = {theta_p*RTF*1e3:.1f} mV;  Delta E_p = {(theta_pv-theta_p)*RTF*1e3:.1f} mV a {T} K  (2,22 RT/F)")

plt.figure(figsize=(6, 4))
plt.plot(-theta * RTF, -chi, color="#1D4ED8")      # espelhado: oxidação para cima, E crescente
plt.xlabel("E − E°′ (V)"); plt.ylabel("i / [FAc*(FvD/RT)^½]"); plt.grid(alpha=0.3)
plt.title("o voltamograma adimensional"); plt.show()

**O número puro.** O pico vale 0,4463 e não depende de $D$, $v$ nem $c^*$: é só Fick com Nernst. Voltando às unidades, $i_p = 0{,}446\,FAc^*\sqrt{FvD/RT}$: a equação de Randles–Ševčík. Confira também que $\Delta E_p = 2{,}22\,RT/F$ (58 mV a 303 K, 57 mV a 25 °C).

## 2. Os picos medidos: a reta de $i_p$ contra $\sqrt{v}$

`dados/elgrishi2018_figS4_picos.csv` tem os oito pares de picos da Fig. S4.1 (0,05 a 10 V/s), digitalizados. Na figura do SI a corrente catódica é positiva; aqui usamos os dois ramos em módulo, porque o que importa é a inclinação. O SI ajustou retas **pela origem** e obteve 97,4 e −102,5 µA·s^½·V^−½.

In [ ]:
picos = pd.read_csv("dados/elgrishi2018_figS4_picos.csv")
rv = picos["raiz_v"].values
ip_a = np.abs(picos["ip_anodico_uA"].values) * 1e-6     # A
ip_c = np.abs(picos["ip_catodico_uA"].values) * 1e-6
s_a = ...        # COMPLETE: inclinação da reta pela origem, ramo anódico
s_c = ...        # COMPLETE: idem, ramo catódico
D_a = ...        # COMPLETE: D a partir da inclinação (Randles-Ševčík com 0,446)
D_c = ...        # COMPLETE
s_prev = 0.446 * F * A * c_bulk * np.sqrt(F * D_lit / (R * T))
print(f"inclinações medidas: {s_a*1e6:.1f} e {s_c*1e6:.1f} uA s^1/2 V^-1/2  (SI: 97,4 e 102,5)")
print(f"D medido = {D_a*1e5:.2f} e {D_c*1e5:.2f} x 1e-5 cm2/s   (SI: 2,7 e 2,9; literatura: {D_lit*1e5:.2f})")
print(f"inclinação prevista com D da literatura: {s_prev*1e6:.1f} uA s^1/2 V^-1/2")

x = np.linspace(0, 3.3, 10)
plt.figure(figsize=(6, 4))
plt.plot(rv, ip_a*1e6, "o", color="#1D4ED8", label="pico anódico (medido)")
plt.plot(rv, ip_c*1e6, "s", mfc="none", color="#1D4ED8", label="pico catódico (medido)")
plt.plot(x, s_prev*x*1e6, "--", color="#A16207", label="previsto, D da literatura")
plt.xlabel("v^½ (V^½ s^−½)"); plt.ylabel("|i_p| (µA)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 3. A curva simulada sobre a medida, a 10 V/s

`dados/elgrishi2018_figS3_10Vs.csv` é a curva externa da Fig. S3.2 (10 V/s), digitalizada, com o zero de corrente no início da varredura. Vamos pôr a simulação da seção 1 em unidades reais, com o $D$ da literatura e **sem ajuste**, e comparar.

In [ ]:
cv = pd.read_csv("dados/elgrishi2018_figS3_10Vs.csv")
ida = cv[cv.ramo == "ida"]; volta = cv[cv.ramo == "volta"]
# o topo digitalizado é chato: o potencial de pico é o centro do platô
E_pa = ida.E_V[ida.i_uA >= ida.i_uA.max() - 0.5].mean()
E_pc = volta.E_V[volta.i_uA <= volta.i_uA.min() + 0.5].mean()
E_meio = 0.5 * (E_pa + E_pc)
v = 10.0                                              # V/s
i_ref = ...                 # COMPLETE: F A c* (F v D / RT)^1/2
E_sim = E_meio - theta * RTF
i_sim = chi * i_ref * 1e6   # uA
print(f"pico simulado a 10 V/s: {i_sim.max():.0f} uA;  medido: {ida.i_uA.max():.0f} uA")
print(f"Delta E_p medido: {(E_pa - E_pc)*1e3:.0f} mV;  nernstiano: {2.22*RTF*1e3:.0f} mV")
plt.figure(figsize=(6, 4))
plt.plot(ida.E_V, ida.i_uA, color="#1D4ED8", label="medida (ida)")
plt.plot(volta.E_V, volta.i_uA, color="#1D4ED8", alpha=0.6, label="medida (volta)")
plt.plot(E_sim, i_sim, "--", color="#A16207", label="simulada, D da literatura")
plt.xlim(-0.45, 0.35); plt.xlabel("E (V contra Fc⁺/Fc)"); plt.ylabel("i (µA)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

A forma e o pico batem a cerca de 10 %; $\Delta E_p$ não. O SI diz, no Módulo 3, que mesmo com 0,25 M de eletrólito a queda ôhmica aparece nas varreduras rápidas: $iR_u$ desloca os dois picos para fora. É o custo do eletrodo ideal que a aula supôs, e é o que a Aula 03 vai medir.

## 4. Disco rotatório: Levich e Koutecký–Levich

**Atenção: os dados abaixo são fornecidos e simulados**, porque não há medida de disco rotatório nas fontes desta aula. Foram gerados com a equação de Koutecký–Levich para um par com $D = 7{,}3 \times 10^{-6}$ cm²/s, $\nu = 0{,}010$ cm²/s (valores típicos de um íon em água), $c^* = 1$ mM, $A = 0{,}196$ cm² (disco de 5 mm), $i_k = 1{,}2$ mA, e ruído de 1 %. A tarefa é recuperar $D$ e $i_k$ a partir deles, como se fossem medidos.

In [ ]:
rng = np.random.default_rng(2026)
D_sim, nu_sim, c_sim, A_sim, ik_sim = 7.3e-6, 0.010, 1.0e-6, 0.196, 1.2e-3
rpm = np.array([400, 625, 900, 1225, 1600, 2025, 2500])
omega = 2 * np.pi * rpm / 60.0                               # rad/s
iL_sim = 0.620 * F * A_sim * D_sim**(2/3) * nu_sim**(-1/6) * c_sim * np.sqrt(omega)
i_medido = 1.0 / (1.0/ik_sim + 1.0/iL_sim) * (1 + 0.01 * rng.standard_normal(rpm.size))
disco = pd.DataFrame({"rpm": rpm, "omega_rad_s": omega, "i_A": i_medido})
disco

In [ ]:
x = 1.0 / np.sqrt(omega)
coef = np.polyfit(x, 1.0 / i_medido, 1)     # reta de 1/i contra 1/omega^1/2
inclinacao, intercepto = coef
ik_rec = ...        # COMPLETE: a corrente cinética, do intercepto
B = ...             # COMPLETE: a constante de Levich, da inclinação
D_rec = ...         # COMPLETE: D a partir de B (B = 0,620 F A D^2/3 nu^-1/6 c*)
print(f"i_k recuperado = {ik_rec*1e3:.2f} mA  (gerado com {ik_sim*1e3:.2f})")
print(f"D recuperado = {D_rec*1e6:.2f} x 1e-6 cm2/s  (gerado com {D_sim*1e6:.2f})")
plt.figure(figsize=(6, 4))
xx = np.linspace(0, x.max()*1.05, 10)
plt.plot(x, 1.0/i_medido, "o", color="#1D4ED8", label="dados fornecidos (simulados)")
plt.plot(xx, np.polyval(coef, xx), "--", color="#A16207", label="ajuste de Koutecký-Levich")
plt.xlabel("ω^−½ (s^½)"); plt.ylabel("1/i (A⁻¹)"); plt.legend(); plt.grid(alpha=0.3); plt.show()

O $D$ volta a 2 %; o $i_k$, não: o intercepto de uma reta extrapolada até $\omega \to \infty$ amplifica o ruído de 1 % dos pontos. É uma limitação real de Koutecký–Levich, e vale a pena medir quanto.

## Para discutir

1. Na seção 1, o que acontece com $\Delta E_p$ se a condição de Nernst for trocada por Butler–Volmer com um $j_0$ finito? (Dica: é o que o artigo chama de irreversibilidade eletroquímica.)
2. Na seção 2, os picos lentos ficam um pouco acima da reta pela origem (o ajuste livre dá intercepto positivo). Que corrente, que não é de difusão, cresce com $v$ e se soma ao pico?
3. Na seção 3, estime $R_u$ a partir do deslocamento dos picos a 10 V/s. É compatível com os 215 Ω que o Módulo 2 mede a 50 mM?
4. Na seção 4, se a reta de Koutecký–Levich passasse pela origem, o que isso diria da cinética? E por que um disco rotatório mede $i_k$ melhor do que uma voltametria cíclica?